# 8  Explaining predictions

**Goal:** see which inputs a model relies on overall, explain a single decision, test what would change it, and write the explanation in plain language.

**Data:** the late applications, without the leaking input.

**The data used here:**
- `late_applications.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/late_applications.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/late_applications.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  The model

In [ ]:
#@title Step 1  Train the model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
df = pd.read_csv(DATA + "late_applications.csv", parse_dates=["documents_uploaded_at", "submitted_at"])
df["upload_gap_days"] = ((df["submitted_at"] - df["documents_uploaded_at"]).dt.total_seconds() / 86400).round(1)
df["submitted_thursday"] = (df["submitted_at"].dt.day_of_week == 3).astype(int)
INPUTS = ["service", "channel", "number_of_documents", "documents_complete", "medical_fitness_document_required", "additional_document_review_required", "queue_at_submission", "upload_gap_days", "submitted_thursday"]
X = pd.get_dummies(df[INPUTS], dtype=int)
for c in ("documents_complete", "medical_fitness_document_required", "additional_document_review_required"): X = X.drop(columns=f"{c}_No")
y = df["late"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42).fit(X_train, y_train)
big((f"{model.score(X_test, y_test):.1%}", "right on the test data"))

## Step 2  Which inputs matter for the whole model?

In [ ]:
#@title Step 2  Feature importance
pi = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(8)
bars(imp, "How much worse the model gets when each input is shuffled", highlight=imp.index[0], fmt="{:.3f}")

In [ ]:
#@title Step 2b  Does this explain why one application will be late?  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "Yes", "No"]
_ = answer(question, {
    "Yes": (False, "Feature importance describes the whole model, averaged over all applications. One application can be late for a completely different reason."),
    "No": (True, "It describes the whole model. For one decision about one person, we need a different method: SHAP, in the next step."),
})

## Step 3  Why this one prediction?

In [ ]:
#@title Step 3  Pick an application and explain it  (pick an application on the right, then click &#9654;)
application = 1 #@param {type:"slider", min:1, max:30, step:1}
try:
    import shap
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "-q", "install", "shap"], check=True); import shap
explainer = shap.TreeExplainer(model)
case = X_test.iloc[[application - 1]]
sv = explainer(case)[0, :, 1]
p = model.predict_proba(case)[0, 1]
big((f"{p:.0%}", "probability of being late", RED if p >= .5 else GREEN), (f"{sv.base_values:.0%}", "average over all applications"))
shap.plots.waterfall(sv, max_display=8)
top = pd.Series(sv.values, index=X.columns).sort_values(key=abs, ascending=False).head(3)
reasons = "; ".join(f"<b>{k} = {case.iloc[0][k]:g}</b> {'raises' if v > 0 else 'lowers'} it by {abs(v):.0%}" for k, v in top.items())
box("How to read it", f"Start at the average ({sv.base_values:.0%}). Each bar pushes this one prediction up (red) or down (blue). The three largest: {reasons}.", "see")

In [ ]:
#@title Step 4  What would change it?  (change the fields on the right, then click &#9654;)
documents_complete = "as it is" #@param ["as it is", "Yes", "No"]
additional_document_review_required = "as it is" #@param ["as it is", "Yes", "No"]
what_if = case.copy()
for col, v in (("documents_complete", documents_complete), ("additional_document_review_required", additional_document_review_required)):
    if v != "as it is":
        what_if[f"{col}_Yes"] = int(v == "Yes")
p2 = model.predict_proba(what_if)[0, 1]
big((f"{p:.0%}", "as it is"), (f"{p2:.0%}", "with your changes", RED if p2 >= .5 else GREEN))
box("Do you see it?", "Changing one input and running the model again shows what the applicant could do: for example, completing the documents. "
    "That is the most useful part of an explanation for a citizen.", "see")

## Step 5  Explain it in plain language

In [ ]:
#@title Step 5  Which message would you send?  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "A: 'The model's score for your application is 0.83.'", "B: 'Your application will probably take longer than usual, because some documents are missing. You can upload them in the app. If you disagree, you can ask the service centre to review it.'", "C: 'Your application has been classified as late by an AI system.'"]
_ = answer(question, {
    "A: 'The model's score for your application is 0.83.'": (False, "A score means nothing to an applicant: no reason, nothing to do, no way to object."),
    "B: 'Your application will probably take longer than usual, because some documents are missing. You can upload them in the app. If you disagree, you can ask the service centre to review it.'": (True, "It gives the main reason (from SHAP), what the person can do (from the what-if) and how to ask a person for a review."),
    "C: 'Your application has been classified as late by an AI system.'": (False, "No reason, nothing the applicant can do, and no way to ask for a review."),
})

In [ ]:
#@title Step 6  And for a manager?  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "A: 'The model uses a random forest with 200 trees and SHAP values.'", "B: 'The model is right in about 9 of 10 test cases. Additional document reviews, medical fitness documents and the queue drive most predicted delays. Staff review every application predicted to be late.'", "C: 'The model is 87.8% accurate.'"]
_ = answer(question, {
    "A: 'The model uses a random forest with 200 trees and SHAP values.'": (False, "This describes the technique, not what the manager needs to decide."),
    "B: 'The model is right in about 9 of 10 test cases. Additional document reviews, medical fitness documents and the queue drive most predicted delays. Staff review every application predicted to be late.'": (True, "How well it works on test data, which inputs drive it (from feature importance), and where a person reviews the output."),
    "C: 'The model is 87.8% accurate.'": (False, "One number, without what drives the decisions or where a person checks."),
})

## Result

In [ ]:
#@title Result
box("Do you see it now?", "Feature importance shows which inputs the model uses: that makes it <b>transparent</b>. SHAP shows why it decided one case: that makes it <b>explainable</b>, "
    "as the UAE AI Ethics Guidelines ask. <b>Now every applicant can get a reason, a way to act, and a way to ask a person.</b>", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: find a case where the explanation reveals a problem in the model (start with the SHAP summary)
shap.plots.beeswarm(explainer(X_test.iloc[:300])[:, :, 1], max_display=10)
# Does any input push predictions in a direction the service team would not expect? Explain that application with the waterfall plot.